In [1]:
import json

file_path = "/kaggle/input/datasets/saumyajitdas1/adhd-pairs-25000-jsonl/adhd_pairs_25000.jsonl"

valid_data = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        sample = json.loads(line)
        if len(sample["original_text"]) > 20 and len(sample["adhd_text"]) > 20:
            valid_data.append(sample)

print("Valid samples:", len(valid_data))

Valid samples: 25000


In [2]:
from datasets import load_dataset

file_path = "/kaggle/input/datasets/saumyajitdas1/adhd-pairs-25000-jsonl/adhd_pairs_25000.jsonl"

dataset = load_dataset("json", data_files=file_path)

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'topic', 'original_text', 'adhd_text', 'grade_level', 'structure_type', 'emoji_density'],
        num_rows: 25000
    })
})


In [3]:
dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)

train_dataset = dataset["train"]
val_dataset = dataset["test"]

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))

Train size: 22500
Validation size: 2500


In [4]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "google/flan-t5-small"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [5]:
max_input_length = 256
max_target_length = 256

def preprocess_function(examples):
    inputs = ["simplify for adhd: " + text for text in examples["original_text"]]
    targets = examples["adhd_text"]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        targets,
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = train_dataset.map(preprocess_function, batched=True)
val_dataset = val_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/22500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

In [6]:
max_input_length = 256
max_target_length = 256

def preprocess_function(examples):
    inputs = ["simplify for adhd: " + text for text in examples["original_text"]]
    targets = examples["adhd_text"]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        targets,
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = train_dataset.map(preprocess_function, batched=True)
val_dataset = val_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/22500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

In [7]:
!pip install -U transformers accelerate

In [8]:
import transformers
print(transformers.__version__)

5.2.0


In [9]:
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    TrainingArguments,
    Trainer
)

training_args = TrainingArguments(
    output_dir="./adhd_t5_model",
    eval_strategy="epoch",      # correct for v5
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.01,
    save_total_limit=2,
    logging_steps=100,
    fp16=True
)

In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.012561,0.011400
2,0.012466,0.011508
3,0.012767,0.011693
4,0.012499,0.011497


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=5628, training_loss=0.01272697388257384, metrics={'train_runtime': 2738.2185, 'train_samples_per_second': 32.868, 'train_steps_per_second': 2.055, 'total_flos': 8365072711680000.0, 'train_loss': 0.01272697388257384, 'epoch': 4.0})

In [15]:
trainer.save_model("/kaggle/working/adhd_t5_model")
tokenizer.save_pretrained("/kaggle/working/adhd_t5_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/adhd_t5_model/tokenizer_config.json',
 '/kaggle/working/adhd_t5_model/tokenizer.json')

In [ ]:
import shutil

shutil.make_archive(
    base_name="/kaggle/working/adhd_t5_model",
    format="zip",
    root_dir="/kaggle/working",
    base_dir="adhd_t5_model"
)

print("Model zipped successfully!")

In [16]:
def simplify(text):
    input_text = "simplify for adhd: " + text
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        inputs["input_ids"],
        max_length=200,
        num_beams=4,
        repetition_penalty=1.2
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


print(simplify("Photosynthesis is the biochemical process by which plants convert sunlight into chemical energy."))

 Photosynthesis made simple! Here’s what you need to know: • It helps explain things around you. • It’s easy once broken down. Let’s go step by step 
